In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [ ]:
df = pd.read_csv("../Data/processed/customer_churn_features.csv")

print("Shape:", df.shape)
df.head()

Shape: (3000, 29)


,Customer_ID,Age,Tenure_Months,Subscription_Plan,Billing_Frequency,Monthly_Charges,Login_Count_30D,Login_Count_7D,Usage_Minutes_30D,Usage_Minutes_7D,...,Churn,Expected_Login_7D,Login_Drop_Percent,Expected_Usage_7D,Usage_Drop_Percent,Support_Friction_Index,Engagement_Score,High_Inactivity,Payment_Risk,Low_Engagement
0,CUST10000,56.0,6,Basic,Monthly,449.04,17,11,262,237,...,1,3.966667,0.0,61.133333,0.000000,67.2,83.6,0,0,0
1,CUST10001,46.0,15,Premium,Annual,1194.07,10,6,209,215,...,0,2.333333,0.0,48.766667,0.000000,0.0,79.9,0,0,0
2,CUST10002,32.0,18,Standard,Monthly,736.96,19,18,209,168,...,0,4.433333,0.0,48.766667,0.000000,76.0,72.0,0,0,0
3,CUST10003,60.0,32,Premium,Annual,1061.74,19,7,223,30,...,1,4.433333,0.0,52.033333,42.344651,8.1,15.5,0,0,0
4,CUST10004,25.0,11,Standard,Monthly,550.54,14,7,233,87,...,1,3.266667,0.0,54.366667,0.000000,16.0,37.2,0,0,0


In [ ]:
#Features & target
X = df.drop(columns=["Customer_ID", "Churn"])
y = df["Churn"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (3000, 27)
Target: (3000,)


In [ ]:
#Categorical/numerical columns
categorical_features = [
    "Subscription_Plan",
    "Billing_Frequency"
]

numerical_features = [
    col for col in X.columns
    if col not in categorical_features
]

print("Categorical:", categorical_features)
print("Numerical:", len(numerical_features))

Categorical: ['Subscription_Plan', 'Billing_Frequency']
Numerical: 25


In [6]:
#Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [7]:
#Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (2400, 27)
Testing data: (600, 27)


In [8]:
#Logistic Regression
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [9]:
#Random Forest
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

rf_model.fit(X_train, y_train)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [10]:
#XGBoost
xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        random_state=42,
        eval_metric="logloss"
    ))
])

xgb_model.fit(X_train, y_train)

print("XGBoost trained successfully!")

XGBoost trained successfully!


In [11]:
#Save models

import joblib

joblib.dump(logistic_model, "../Models/logistic_model.pkl")
joblib.dump(rf_model, "../Models/random_forest_model.pkl")
joblib.dump(xgb_model, "../Models/xgboost_model.pkl")

print("All 3 models saved successfully!")

All 3 models saved successfully!
